## Candidate Pair Generation using Union Blocking

This notebook generates candidate pairs from the PRL-ready personas dataset.

The input file is `PRL_ready_personas_final.csv`, which contains cleaned person mentions, event metadata, role groups, and blocking keys.

The output of this notebook will be a candidate-pair dataset that can be used for similarity scoring and manual review.

In [27]:
# import required libraries
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations


## Loading the Dataset: `PRL_ready_personas_final.csv`

In [28]:
input_path=Path("../data/interim/PRL_ready_personas_final.csv")
persona= pd.read_csv(input_path)
print("The shape of the dataframe is:", persona.shape)
persona.head(5)

The shape of the dataframe is: (47072, 32)


C:\Users\samav\AppData\Local\Temp\ipykernel_20580\3239210702.py:2: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  persona= pd.read_csv(input_path)


,persona_idno,event_idno,original_identifier,source_event_type,source_table,event_type,persona_type,role_group,name,lastname,...,death_year,death_place_clean,resident_in_clean,gender,block_lastname_initial,block_lastname_firstname,block_last4_firstname,block_lastname_birthyear,matching_evidence_level,cleaning_issue_flags
0,persona-1,bautizo-1,APAucará-LB-L001_B001,Baptism,bautismos,Bautizo,baptized,main_life_event_person,domingo,ayquipa,...,NaN,unknown,unknown,male,ayquipa_d,ayquipa_domingo,ayqu_domingo,ayquipa_1790,high,no_major_issue
1,persona-2,bautizo-1,APAucará-LB-L001_B001,Baptism,bautismos,Bautizo,father,family_relation,lucas,ayquipa,...,NaN,unknown,unknown,male,ayquipa_l,ayquipa_lucas,ayqu_lucas,NaN,medium,no_major_issue
2,persona-3,bautizo-1,APAucará-LB-L001_B001,Baptism,bautismos,Bautizo,mother,family_relation,sevastiana,quispe,...,NaN,unknown,unknown,female,quispe_s,quispe_sevastiana,quis_sevastiana,NaN,medium,no_major_issue
3,persona-4,bautizo-1,APAucará-LB-L001_B001,Baptism,bautismos,Bautizo,godfather,godparent,vicente,guamani,...,NaN,unknown,unknown,male,guamani_v,guamani_vicente,guam_vicente,NaN,medium,no_major_issue
4,persona-5,bautizo-2,APAucará-LB-L001_B002,Baptism,bautismos,Bautizo,baptized,main_life_event_person,dominga,lulia,...,NaN,unknown,unknown,female,lulia_d,lulia_dominga,luli_dominga,lulia_1790,high,no_major_issue


## Step 2: Inspect Blocking Keys

Before generating candidate pairs, we first inspect the blocking columns.

The purpose is to check:
- how many non-empty blocking values exist
- how many unique blocks each rule creates
- whether any block is too large
- whether blank blocking values are being skipped correctly

This helps prevent unnecessary pair generation from weak or oversized blocks.

In [29]:
# Define the blocking columns used for union blocking

blocking_columns = [
    "block_lastname_initial",
    "block_lastname_firstname",
    "block_last4_firstname",
    "block_lastname_birthyear"
]

blocking_columns

['block_lastname_initial',
 'block_lastname_firstname',
 'block_last4_firstname',
 'block_lastname_birthyear']

Check missing values and blank values

In [30]:
# Check missing and blank values in each blocking column

for col in blocking_columns:
    missing_count = persona[col].isna().sum()
    blank_count = (persona[col].fillna("") == "").sum()
    unique_count = persona[col].nunique(dropna=True)

    print(f"\nBlocking column: {col}")
    print(f"Missing values: {missing_count}")
    print(f"Blank values: {blank_count}")
    print(f"Unique non-null values: {unique_count}")


Blocking column: block_lastname_initial
Missing values: 0
Blank values: 0
Unique non-null values: 8014

Blocking column: block_lastname_firstname
Missing values: 0
Blank values: 0
Unique non-null values: 19866

Blocking column: block_last4_firstname
Missing values: 0
Blank values: 0
Unique non-null values: 17531

Blocking column: block_lastname_birthyear
Missing values: 38476
Blank values: 38476
Unique non-null values: 5323


Check largest blocks

In [31]:
# Find the largest blocks for each blocking column

for col in blocking_columns:
    print(f"\nTop 10 largest blocks for: {col}")

    block_sizes = (
        persona[col]
        .fillna("")
        .value_counts()
    )

    # Remove blank block values
    block_sizes = block_sizes[block_sizes.index != ""]

    print(block_sizes.head(10))


Top 10 largest blocks for: block_lastname_initial
block_lastname_initial
quispe_m     455
huamani_m    377
ramos_m      267
condori_m    260
quispe_j     260
quispe_s     239
quispe_c     235
sanches_m    227
quispe_p     217
ramos_j      194
Name: count, dtype: int64

Top 10 largest blocks for: block_lastname_firstname
block_lastname_firstname
quispe_maria       85
mancco_manuel      81
huamani_maria      79
ramos_juana        74
quispe_simon       72
tito_mariano       71
huamani_mariano    70
condori_maria      64
no conocido_no     63
ramos_juan         63
Name: count, dtype: int64

Top 10 largest blocks for: block_last4_firstname
block_last4_firstname
hual_mariano    128
sanc_maria       93
huam_maria       86
quis_maria       85
manc_manuel      83
huam_mariano     76
ramo_juana       74
sanc_juan        73
quis_simon       72
tito_mariano     71
Name: count, dtype: int64

Top 10 largest blocks for: block_lastname_birthyear
block_lastname_birthyear
huamani_1893    22
quispe_1894

In [32]:
# Estimate how many candidate pairs each blocking rule may generate

def estimate_pairs_from_blocks(persona, block_col):
    """
    Estimate the number of record pairs generated by one blocking column.

    If a block has n records, the number of pairs is:
    n * (n - 1) / 2
    """
    block_sizes = (
        persona[block_col]
        .fillna("")
        .value_counts()
    )

    # Remove blank blocks
    block_sizes = block_sizes[block_sizes.index != ""]

    estimated_pairs = ((block_sizes * (block_sizes - 1)) // 2).sum()

    return len(block_sizes), estimated_pairs


for col in blocking_columns:
    num_blocks, estimated_pairs = estimate_pairs_from_blocks(persona, col)

    print(f"\nBlocking column: {col}")
    print(f"Number of non-empty blocks: {num_blocks}")
    print(f"Estimated candidate pairs before deduplication: {estimated_pairs:,}")


Blocking column: block_lastname_initial
Number of non-empty blocks: 8014
Estimated candidate pairs before deduplication: 1,207,499

Blocking column: block_lastname_firstname
Number of non-empty blocks: 19866
Estimated candidate pairs before deduplication: 185,940

Blocking column: block_last4_firstname
Number of non-empty blocks: 17531
Estimated candidate pairs before deduplication: 229,611

Blocking column: block_lastname_birthyear
Number of non-empty blocks: 5323
Estimated candidate pairs before deduplication: 8,382


In [33]:
# Maximum block size allowed for candidate-pair generation

MAX_BLOCK_SIZE = 500

print("Maximum block size allowed:", MAX_BLOCK_SIZE)

Maximum block size allowed: 500


In [34]:
for col in blocking_columns:
    num_blocks, estimated_pairs = estimate_pairs_from_blocks(persona, col)

    print(f"\nBlocking column: {col}")
    print(f"Number of non-empty blocks: {num_blocks}")
    print(f"Estimated candidate pairs before deduplication: {estimated_pairs:,}")


Blocking column: block_lastname_initial
Number of non-empty blocks: 8014
Estimated candidate pairs before deduplication: 1,207,499

Blocking column: block_lastname_firstname
Number of non-empty blocks: 19866
Estimated candidate pairs before deduplication: 185,940

Blocking column: block_last4_firstname
Number of non-empty blocks: 17531
Estimated candidate pairs before deduplication: 229,611

Blocking column: block_lastname_birthyear
Number of non-empty blocks: 5323
Estimated candidate pairs before deduplication: 8,382


## Step 3: Generate Candidate Pairs using Union Blocking

In this step, candidate pairs are generated from all four blocking keys.

A pair is kept if it appears under at least one blocking rule.

If the same pair appears under multiple blocking rules, it is stored only once, and the matched blocking rules are recorded in `blocking_rules_matched`.

We also skip:
- blank blocking values
- blocks with only one record
- very large blocks above the maximum block-size limit
- pairs from the same event

In [35]:
# Blocking rules used for union blocking

blocking_rules = {
    "rule1_lastname_initial": "block_lastname_initial",
    "rule2_lastname_firstname": "block_lastname_firstname",
    "rule3_last4_firstname": "block_last4_firstname",
    "rule4_lastname_birthyear": "block_lastname_birthyear"
}

# Maximum block size allowed
# Very large blocks can create too many weak candidate pairs
MAX_BLOCK_SIZE = 500

blocking_rules

{'rule1_lastname_initial': 'block_lastname_initial',
 'rule2_lastname_firstname': 'block_lastname_firstname',
 'rule3_last4_firstname': 'block_last4_firstname',
 'rule4_lastname_birthyear': 'block_lastname_birthyear'}

In [36]:
def generate_candidate_pairs(persona1,block_col,rule_name,max_block_size):
     """
        Generate candidate pairs for one blocking rule.

        Parameters:
        persona : pandas DataFrame
            PRL-ready personas dataset.
        block_col : str
            Blocking column used to group records.
        rule_name : str
            Name of the blocking rule.
        max_block_size : int
            Maximum allowed number of records in one block.

        Returns:
        pairs : list
            List of candidate pair dictionaries.
        skipped_blocks : list
            List of blocks skipped because they were too large.
    """
     pairs=[]
     skipped_pairs=[]

     # keep the non-empty blocks
     temp=persona1[(persona1[block_col].notna()) & (persona1[block_col].astype(str).str.strip()!="")].copy()

     # group records by blocking_values
     grouped = temp.groupby(block_col)

     for block_value,group in grouped:
          block_size = len(group)
          if block_size<2:
               continue
          
          # skip large blocks
          if block_size > max_block_size:
               skipped_pairs.append(
                    {"rule_name":rule_name,
                     "block_value": block_value,
                     "block_size": block_size,
                     "block_column": block_col
                    }
               )
               continue
    
    # keep columns for pair generations

          group_col = group[["persona_idno","event_idno"]].to_dict('records')

     # generate all possible pairs within the block
          for record_1,record_2 in combinations(group_col, 2):
          
               persona_1 = record_1["persona_idno"]
               event_1 = record_1["event_idno"]
               persona_2 = record_2["persona_idno"]
               event_2 = record_2["event_idno"]

               if event_1 == event_2:
                    continue
               pair_id1,pair_id2=sorted([persona_1,persona_2])
               pairs.append({
                    "persona_idno_1": pair_id1,
                    "persona_idno_2": pair_id2,
                    "blocking_rule": rule_name
               })
     return pairs, skipped_pairs
 

Generate pairs from all blocking rules

In [37]:
all_pairs = []
all_skipped_blocks = []

for rule_name, block_col in blocking_rules.items():

    print(f"Generating candidate pairs for {rule_name} using {block_col}...")

    rule_pairs, skipped_blocks = generate_candidate_pairs(
        persona1=persona,
        block_col=block_col,
        rule_name=rule_name,
        max_block_size=MAX_BLOCK_SIZE
    )

    all_pairs.extend(rule_pairs)
    all_skipped_blocks.extend(skipped_blocks)

    print(f"Pairs generated from {rule_name}: {len(rule_pairs):,}")
    print(f"Large blocks skipped from {rule_name}: {len(skipped_blocks):,}")

Generating candidate pairs for rule1_lastname_initial using block_lastname_initial...
Pairs generated from rule1_lastname_initial: 1,206,405
Large blocks skipped from rule1_lastname_initial: 0
Generating candidate pairs for rule2_lastname_firstname using block_lastname_firstname...
Pairs generated from rule2_lastname_firstname: 185,785
Large blocks skipped from rule2_lastname_firstname: 0
Generating candidate pairs for rule3_last4_firstname using block_last4_firstname...
Pairs generated from rule3_last4_firstname: 229,449
Large blocks skipped from rule3_last4_firstname: 0
Generating candidate pairs for rule4_lastname_birthyear using block_lastname_birthyear...
Pairs generated from rule4_lastname_birthyear: 8,381
Large blocks skipped from rule4_lastname_birthyear: 0


Convert generated pairs into DataFrame

In [38]:
candidate_pairs_raw = pd.DataFrame(all_pairs)

print("Raw candidate pairs before deduplication:")
print(candidate_pairs_raw.shape)

candidate_pairs_raw.head()

Raw candidate pairs before deduplication:
(1630020, 3)


,persona_idno_1,persona_idno_2,blocking_rule
0,persona-4835,persona-5977,rule1_lastname_initial
1,persona-4835,persona-6001,rule1_lastname_initial
2,persona-4835,persona-6004,rule1_lastname_initial
3,persona-4835,persona-6232,rule1_lastname_initial
4,persona-4835,persona-6384,rule1_lastname_initial


Apply union blocking and deduplicate pairs

In [39]:
candidate_pairs_union = (
    candidate_pairs_raw
    .groupby(["persona_idno_1", "persona_idno_2"])["blocking_rule"]
    .agg(lambda x: "|".join(sorted(set(x))))
    .reset_index()
)

candidate_pairs_union["blocking_rule_count"] = (
    candidate_pairs_union["blocking_rule"]
    .str.split("|")
    .apply(len)
)

candidate_pairs_union = candidate_pairs_union.rename(
    columns={"blocking_rule": "blocking_rules_matched"}
)

print("Candidate pairs after union blocking and deduplication:")
print(candidate_pairs_union.shape)

candidate_pairs_union.head()

Candidate pairs after union blocking and deduplication:
(1257724, 4)


,persona_idno_1,persona_idno_2,blocking_rules_matched,blocking_rule_count
0,persona-10,persona-10078,rule1_lastname_initial,1
1,persona-10,persona-1008,rule1_lastname_initial,1
2,persona-10,persona-10081,rule1_lastname_initial,1
3,persona-10,persona-10320,rule1_lastname_initial,1
4,persona-10,persona-10366,rule1_lastname_initial,1


In [40]:
candidate_pairs_union.value_counts("blocking_rule_count")

blocking_rule_count
1    1071363
3     185635
2        576
4        150
Name: count, dtype: int64

In [41]:
candidate_pairs_union.value_counts("blocking_rules_matched")

blocking_rules_matched
rule1_lastname_initial                                                                            1020044
rule1_lastname_initial|rule2_lastname_firstname|rule3_last4_firstname                              185635
rule3_last4_firstname                                                                               43664
rule4_lastname_birthyear                                                                             7655
rule1_lastname_initial|rule4_lastname_birthyear                                                       576
rule1_lastname_initial|rule2_lastname_firstname|rule3_last4_firstname|rule4_lastname_birthyear        150
Name: count, dtype: int64

In [42]:
print("Number of final candidate pairs:", len(candidate_pairs_union))

print("\nBlocking rule count distribution:")
print(candidate_pairs_union["blocking_rule_count"].value_counts().sort_index())

print("\nSample candidate pairs:")
candidate_pairs_union.head(10)

Number of final candidate pairs: 1257724

Blocking rule count distribution:
blocking_rule_count
1    1071363
2        576
3     185635
4        150
Name: count, dtype: int64

Sample candidate pairs:


,persona_idno_1,persona_idno_2,blocking_rules_matched,blocking_rule_count
0,persona-10,persona-10078,rule1_lastname_initial,1
1,persona-10,persona-1008,rule1_lastname_initial,1
2,persona-10,persona-10081,rule1_lastname_initial,1
3,persona-10,persona-10320,rule1_lastname_initial,1
4,persona-10,persona-10366,rule1_lastname_initial,1
5,persona-10,persona-1040,rule1_lastname_initial|rule2_lastname_firstnam...,3
6,persona-10,persona-10494,rule1_lastname_initial,1
7,persona-10,persona-10507,rule1_lastname_initial,1
8,persona-10,persona-10601,rule1_lastname_initial,1
9,persona-10,persona-10883,rule1_lastname_initial,1


In [43]:
print("Number of raw candidate pairs:", candidate_pairs_raw.shape[0])
print("Number of final candidate pairs:", candidate_pairs_union.shape[0])
print("Number of blocking rules matched per pair:")
print(candidate_pairs_union["blocking_rule_count"].value_counts().sort_index())

Number of raw candidate pairs: 1630020
Number of final candidate pairs: 1257724
Number of blocking rules matched per pair:
blocking_rule_count
1    1071363
2        576
3     185635
4        150
Name: count, dtype: int64


## Step 4: Add Persona Details to Candidate Pairs

The candidate-pair table currently contains only the two persona IDs and blocking-rule information.

In this step, we merge selected columns from the original PRL-ready personas file so that each pair contains information for both records.

This will allow us to compare names, gender, years, places, and roles in the next step.

In [44]:
# Columns needed for pairwise comparison

persona_detail_columns = [
    "persona_idno",
    "event_idno",
    "original_identifier",
    "source_event_type",
    "persona_type",
    "role_group",

    "name",
    "lastname",
    "name_clean",
    "lastname_clean",
    "full_name_clean",
    "first_name_clean",

    "gender",

    "event_year",
    "birth_year",
    "death_year",

    "event_place_clean",
    "birth_place_clean",
    "death_place_clean",
    "resident_in_clean",

    "matching_evidence_level",
    "cleaning_issue_flags"
]

persona_details = persona[persona_detail_columns].copy()

print("Persona details shape:", persona_details.shape)
persona_details.head()

Persona details shape: (47072, 22)


,persona_idno,event_idno,original_identifier,source_event_type,persona_type,role_group,name,lastname,name_clean,lastname_clean,...,gender,event_year,birth_year,death_year,event_place_clean,birth_place_clean,death_place_clean,resident_in_clean,matching_evidence_level,cleaning_issue_flags
0,persona-1,bautizo-1,APAucará-LB-L001_B001,Baptism,baptized,main_life_event_person,domingo,ayquipa,domingo,ayquipa,...,male,1790.0,1790.0,NaN,pampamarca,unknown,unknown,unknown,high,no_major_issue
1,persona-2,bautizo-1,APAucará-LB-L001_B001,Baptism,father,family_relation,lucas,ayquipa,lucas,ayquipa,...,male,1790.0,NaN,NaN,pampamarca,unknown,unknown,unknown,medium,no_major_issue
2,persona-3,bautizo-1,APAucará-LB-L001_B001,Baptism,mother,family_relation,sevastiana,quispe,sevastiana,quispe,...,female,1790.0,NaN,NaN,pampamarca,unknown,unknown,unknown,medium,no_major_issue
3,persona-4,bautizo-1,APAucará-LB-L001_B001,Baptism,godfather,godparent,vicente,guamani,vicente,guamani,...,male,1790.0,NaN,NaN,pampamarca,unknown,unknown,unknown,medium,no_major_issue
4,persona-5,bautizo-2,APAucará-LB-L001_B002,Baptism,baptized,main_life_event_person,dominga,lulia,dominga,lulia,...,female,1790.0,1790.0,NaN,pampamarca,unknown,unknown,unknown,high,no_major_issue


In [45]:
# adding two versions for persona_1 and persona_2
persona_details_1 = persona_details.add_suffix("_1")
persona_details_2 = persona_details.add_suffix("_2")

Merge details into candidate pairs

In [46]:
# adding details for first persona
candidate_pairs_with_details = candidate_pairs_union.merge(
    persona_details_1,
    on="persona_idno_1",
    how="left"
)

# adding details for second persona
candidate_pairs_with_details = candidate_pairs_with_details.merge(
    persona_details_2,
    on="persona_idno_2",
    how="left"
)

print("Candidate pairs with details shape:")
print(candidate_pairs_with_details.shape)

candidate_pairs_with_details.head()

Candidate pairs with details shape:
(1257724, 46)


,persona_idno_1,persona_idno_2,blocking_rules_matched,blocking_rule_count,event_idno_1,original_identifier_1,source_event_type_1,persona_type_1,role_group_1,name_1,...,gender_2,event_year_2,birth_year_2,death_year_2,event_place_clean_2,birth_place_clean_2,death_place_clean_2,resident_in_clean_2,matching_evidence_level_2,cleaning_issue_flags_2
0,persona-10,persona-10078,rule1_lastname_initial,1,bautizo-3,APAucará-LB-L001_B003,Baptism,father,family_relation,jacinto,...,male,1838.0,NaN,NaN,pampamarca santa iglesia,unknown,unknown,unknown,medium,no_major_issue
1,persona-10,persona-1008,rule1_lastname_initial,1,bautizo-3,APAucará-LB-L001_B003,Baptism,father,family_relation,jacinto,...,female,1793.0,1793.0,NaN,unknown,unknown,unknown,unknown,medium,no_major_issue
2,persona-10,persona-10081,rule1_lastname_initial,1,bautizo-3,APAucará-LB-L001_B003,Baptism,father,family_relation,jacinto,...,male,1838.0,NaN,NaN,ishua santa iglesia,ishua,unknown,unknown,high,no_major_issue
3,persona-10,persona-10320,rule1_lastname_initial,1,bautizo-3,APAucará-LB-L001_B003,Baptism,father,family_relation,jacinto,...,male,1837.0,NaN,NaN,pampamarca santa iglesia,unknown,unknown,unknown,medium,no_major_issue
4,persona-10,persona-10366,rule1_lastname_initial,1,bautizo-3,APAucará-LB-L001_B003,Baptism,father,family_relation,jacinto,...,male,1896.0,NaN,NaN,pampamarca santa iglesia viceparroquial,unknown,unknown,unknown,medium,no_major_issue


Validating the merge checking whether any pairs failed to merge with persona details

In [47]:
missing_persona_1 = candidate_pairs_with_details["event_idno_1"].isna().sum()
missing_persona_2 = candidate_pairs_with_details["event_idno_2"].isna().sum()

print("Pairs missing details for persona 1:", missing_persona_1)
print("Pairs missing details for persona 2:", missing_persona_2)

Pairs missing details for persona 1: 0
Pairs missing details for persona 2: 0


Validating whether the same event pairs were removed or not

In [48]:
same_event_pairs = (
    candidate_pairs_with_details["event_idno_1"] ==
    candidate_pairs_with_details["event_idno_2"]
).sum()

print("Number of same-event candidate pairs:", same_event_pairs)


Number of same-event candidate pairs: 0


Viewing the useful columns 

In [49]:
preview_columns = [
    "persona_idno_1",
    "persona_idno_2",
    "blocking_rules_matched",
    "blocking_rule_count",

    "name_clean_1",
    "name_clean_2",
    "lastname_clean_1",
    "lastname_clean_2",

    "gender_1",
    "gender_2",

    "birth_year_1",
    "birth_year_2",
    "event_year_1",
    "event_year_2",

    "source_event_type_1",
    "source_event_type_2",
    "persona_type_1",
    "persona_type_2",
    "role_group_1",
    "role_group_2"
]

candidate_pairs_with_details[preview_columns].head(10)

,persona_idno_1,persona_idno_2,blocking_rules_matched,blocking_rule_count,name_clean_1,name_clean_2,lastname_clean_1,lastname_clean_2,gender_1,gender_2,birth_year_1,birth_year_2,event_year_1,event_year_2,source_event_type_1,source_event_type_2,persona_type_1,persona_type_2,role_group_1,role_group_2
0,persona-10,persona-10078,rule1_lastname_initial,1,jacinto,jose,quispe,quispe,male,male,NaN,NaN,1790.0,1838.0,Baptism,Baptism,father,father,family_relation,family_relation
1,persona-10,persona-1008,rule1_lastname_initial,1,jacinto,juliana,quispe,quispe,male,female,NaN,1793.0,1790.0,1793.0,Baptism,Baptism,father,baptized,family_relation,main_life_event_person
2,persona-10,persona-10081,rule1_lastname_initial,1,jacinto,jose,quispe,quispe,male,male,NaN,NaN,1790.0,1838.0,Baptism,Baptism,father,baptized,family_relation,main_life_event_person
3,persona-10,persona-10320,rule1_lastname_initial,1,jacinto,julian,quispe,quispe,male,male,NaN,NaN,1790.0,1837.0,Baptism,Baptism,father,father,family_relation,family_relation
4,persona-10,persona-10366,rule1_lastname_initial,1,jacinto,julian,quispe,quispe,male,male,NaN,NaN,1790.0,1896.0,Baptism,Baptism,father,father,family_relation,family_relation
5,persona-10,persona-1040,rule1_lastname_initial|rule2_lastname_firstnam...,3,jacinto,jacinto,quispe,quispe,male,male,NaN,NaN,1790.0,1793.0,Baptism,Baptism,father,father,family_relation,family_relation
6,persona-10,persona-10494,rule1_lastname_initial,1,jacinto,juana,quispe,quispe,male,female,NaN,NaN,1790.0,1896.0,Baptism,Baptism,father,mother,family_relation,family_relation
7,persona-10,persona-10507,rule1_lastname_initial,1,jacinto,juana,quispe,quispe,male,female,NaN,NaN,1790.0,1896.0,Baptism,Baptism,father,godmother,family_relation,godparent
8,persona-10,persona-10601,rule1_lastname_initial,1,jacinto,jervacio,quispe,quispe,male,unknown,NaN,NaN,1790.0,1896.0,Baptism,Baptism,father,father,family_relation,family_relation
9,persona-10,persona-10883,rule1_lastname_initial,1,jacinto,jervacio,quispe,quispe,male,unknown,NaN,NaN,1790.0,1899.0,Baptism,Baptism,father,father,family_relation,family_relation


In [50]:
print("Shape of candidate pairs with details:", candidate_pairs_with_details.shape)


Shape of candidate pairs with details: (1257724, 46)


## Exporting the candidate pairs

In [51]:
output_path = "../data/interim/candidate_pairs_union.csv"

candidate_pairs_union.to_csv(output_path, index=False)

print("Saved candidate pairs to:", output_path)
print(candidate_pairs_union.shape)


Saved candidate pairs to: ../data/interim/candidate_pairs_union.csv
(1257724, 4)


In [52]:
output_path = "../data/interim/candidate_pairs_with_details.parquet"

candidate_pairs_with_details.to_parquet(output_path, index=False)

print("Saved candidate pairs with details to:", output_path)
print(candidate_pairs_with_details.shape)

Saved candidate pairs with details to: ../data/interim/candidate_pairs_with_details.parquet
(1257724, 46)
